System information (for reproducibility):

In [1]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 10 × Apple M1 Pro
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m1)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 8 virtual cores)


Load packages:

In [2]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Desktop/UCLA/Biostat 257/257-hw/hw3`


Status `~/Desktop/UCLA/Biostat 257/257-hw/hw3/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
  [31c24e10] Distributions v0.25.125
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


In [3]:
using LinearAlgebra, Random
using BenchmarkTools, Distributions

Consider a linear mixed effects model
$$
    \mathbf{Y}_i = \mathbf{X}_i \boldsymbol{\beta} + \mathbf{Z}_i \boldsymbol{\gamma}_i + \boldsymbol{\epsilon}_i, \quad i=1,\ldots,n,
$$
where   
- $\mathbf{Y}_i \in \mathbb{R}^{n_i}$ is the response vector of $i$-th individual,  
- $\mathbf{X}_i \in \mathbb{R}^{n_i \times p}$ is the fixed effect predictor matrix of $i$-th individual,  
- $\mathbf{Z}_i \in \mathbb{R}^{n_i \times q}$ is the random effect predictor matrix of $i$-th individual,  
- $\boldsymbol{\epsilon}_i \in \mathbb{R}^{n_i}$ are multivariate normal $N(\mathbf{0}_{n_i},\sigma^2 \mathbf{I}_{n_i})$,  
- $\boldsymbol{\beta} \in \mathbb{R}^p$ are fixed effects, and  
- $\boldsymbol{\gamma}_i \in \mathbb{R}^q$ are random effects assumed to be $N(\mathbf{0}_q, \boldsymbol{\Sigma}_{q \times q})$ independent of $\boldsymbol{\epsilon}_i$.

## Q1. Formula (10 pts)

Write down the log-likelihood of the $i$-th datum $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$ given parameters $(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2)$. 

**Answer:**

Since $\boldsymbol{\gamma}_i \sim N(\mathbf{0}, \boldsymbol{\Sigma})$ and $\boldsymbol{\epsilon}_i \sim N(\mathbf{0}, \sigma^2 \mathbf{I}_{n_i})$ independently:

$$\mathbf{Y}_i = \mathbf{X}_i\boldsymbol{\beta} + \mathbf{Z}_i\boldsymbol{\gamma}_i + \boldsymbol{\epsilon}_i \sim N(\mathbf{X}_i\boldsymbol{\beta},\ \mathbf{V}_i)$$

where the marginal covariance is:
$$\mathbf{V}_i = \mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^\top + \sigma^2 \mathbf{I}_{n_i}$$

$$\boxed{\ell_i(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) = -\frac{n_i}{2}\log(2\pi) - \frac{1}{2}\log|\mathbf{V}_i| - \frac{1}{2}(\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta})^\top \mathbf{V}_i^{-1}(\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta})}$$

with $\mathbf{V}_i = \mathbf{Z}_i\boldsymbol{\Sigma}\mathbf{Z}_i^\top + \sigma^2\mathbf{I}_{n_i}$.

## Q2. Start-up code

Use the following template to define a type `LmmObs` that holds an LMM datum $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$. 

**Answer:**

In [4]:
# define a type that holds LMM datum
struct LmmObs{T <: AbstractFloat}
    # data
    y          :: Vector{T}
    X          :: Matrix{T}
    Z          :: Matrix{T}
    # precomputed scalars/vectors/matrices — ORDER MATTERS
    yty        :: T
    xty        :: Vector{T}
    zty        :: Vector{T}
    xtx        :: Matrix{T}
    ztx        :: Matrix{T}
    ztz        :: Matrix{T}
    # working arrays
    storage_p  :: Vector{T}
    storage_q  :: Vector{T}
    storage_qq :: Matrix{T}
end

# constructor
function LmmObs(y::Vector{T}, X::Matrix{T}, Z::Matrix{T}) where T <: AbstractFloat
    yty        = dot(y, y)
    xty        = transpose(X) * y
    zty        = transpose(Z) * y
    xtx        = transpose(X) * X
    ztx        = transpose(Z) * X
    ztz        = transpose(Z) * Z
    storage_p  = Vector{T}(undef, size(X, 2))
    storage_q  = Vector{T}(undef, size(Z, 2))
    storage_qq = Matrix{T}(undef, size(Z, 2), size(Z, 2))
    # must match struct field order exactly ↓
    LmmObs(y, X, Z, yty, xty, zty, xtx, ztx, ztz, storage_p, storage_q, storage_qq)
end

LmmObs

Write a function, with interface   
```julia
logl!(obs, β, L, σ²)
```
that evaluates the log-likelihood of the $i$-th datum. Here `L` is the lower triangular Cholesky factor from the Cholesky decomposition `Σ=LL'`. Make your code efficient in the $n_i \gg q$ case. Think the intensive longitudinal measurement setting.  

In [5]:
function logl!(
        obs :: LmmObs{T},
        β   :: Vector{T},
        L   :: Matrix{T},
        σ²  :: T) where T <: AbstractFloat
    n, p, q = size(obs.X, 1), size(obs.X, 2), size(obs.Z, 2)

    # Step 1: M = I_q + (1/σ²) L'(Z'Z)L  in storage_qq
    copy!(obs.storage_qq, obs.ztz)
    BLAS.trmm!('L', 'L', 'T', 'N', one(T),      L, obs.storage_qq)  # L'(Z'Z)
    BLAS.trmm!('R', 'L', 'N', 'N', one(T) / σ², L, obs.storage_qq)  # /σ² * L
    @inbounds for j in 1:q
        obs.storage_qq[j, j] += one(T)
    end

    # Step 2: in-place Cholesky of M, log|V| = 2Σlog(Mjj) + n*log(σ²)
    LAPACK.potrf!('L', obs.storage_qq)
    logdetV = n * log(σ²)
    @inbounds for j in 1:q
        logdetV += 2 * log(obs.storage_qq[j, j])
    end

    # Step 3: r'r = yty - 2β'xty + β'(X'X)β  — no n-vector formed
    # storage_p = X'X * β
    copy!(obs.storage_p, obs.xty)                                     # storage_p = X'y
    rtr = obs.yty                                                      # y'y
    BLAS.symv!('U', one(T), obs.xtx, β, zero(T), obs.storage_p)      # xtx*β
    rtr += dot(β, obs.storage_p) - 2 * dot(β, obs.xty)               # + β'Xβ - 2β'X'y

    # Step 4: Z'r = zty - Z'X*β = zty - ztx*β  (q-vector, no n-vector)
    copy!(obs.storage_q, obs.zty)                                     # Z'y
    BLAS.gemv!('N', -one(T), obs.ztx, β, one(T), obs.storage_q)      # -= (Z'X)β

    # storage_q = L'(Z'r)
    BLAS.trmv!('L', 'T', 'N', L, obs.storage_q)
    # storage_q = Mc \ storage_q  (forward solve with Cholesky factor)
    BLAS.trsv!('L', 'N', 'N', obs.storage_qq, obs.storage_q)

    qform = (rtr - dot(obs.storage_q, obs.storage_q) / σ²) / σ²

    # Step 5: log-likelihood
    return -(n / 2) * log(2π) - logdetV / 2 - qform / 2
end

logl! (generic function with 1 method)

**Hint**: This function shouldn't be very long. Mine, obeying 92-character rule, is 30 lines. If you find yourself writing very long code, you're on the wrong track. Think about algorithm (flop count) first then use BLAS functions to reduce memory allocations.

## Q3. Correctness (15 pts)

Compare your result (both accuracy and timing) to the [Distributions.jl](https://juliastats.org/Distributions.jl/stable/multivariate/#Distributions.AbstractMvNormal) package using following data.

In [6]:
Random.seed!(257)

# dimension
n, p, q = 2000, 5, 3
# predictors
X  = [ones(n) randn(n, p - 1)]
Z  = [ones(n) randn(n, q - 1)]
# parameter values
β  = [2.0; -1.0; rand(p - 2)]
σ² = 1.5
Σ  = fill(0.1, q, q) + 0.9I
# generate y
y  = X * β + Z * rand(MvNormal(Σ)) + sqrt(σ²) * randn(n)

# form an LmmObs object
obs = LmmObs(y, X, Z)

LmmObs{Float64}([-1.450910909560209, 1.5185224894450862, 5.265021705624027, 4.485272594164557, 0.694969966642933, 1.7723256696372405, 1.1065838446466518, 3.729166811829607, 4.288899999400642, 2.8241842645202406  …  4.058027151891634, 1.0909724390970443, 0.026692243086209766, -0.8927757653299448, 6.94725248926293, 3.5193020855673436, 4.914007299083773, 2.1610206566690797, 1.857389542694909, 6.513818951020866], [1.0 0.6790633442371218 … 0.5400611947971554 -0.632040682052606; 1.0 1.2456776800889142 … -0.4818455756130373 0.6467830314674976; … ; 1.0 0.0733124748775436 … 0.6125080259511859 0.4181258283983667; 1.0 -1.336609049786048 … -0.18567490803712938 1.0745977099307227], [1.0 -1.0193326822839996 -0.15855601254314888; 1.0 1.7462667837699666 -0.4584376230657152; … ; 1.0 1.4843185594903878 0.42458303115266854; 1.0 0.3791714762820068 0.25150666970865837], 20366.544541546682, [4206.188514588988, -1835.1043159883484, 1339.3508913648914, 396.01166197012594, 1655.5004771264735], [4206.1885145889

This is the standard way to evaluate log-density of a multivariate normal, using the Distributions.jl package. Let's evaluate the log-likelihood of this datum.

In [7]:
μ  = X * β
Ω  = Z * Σ * transpose(Z) +  σ² * I
mvn = MvNormal(μ, Symmetric(Ω)) # MVN(μ, Σ)
logpdf(mvn, y)

-3261.3155007293894

Check that your answer matches that from Distributions.jl

In [8]:
L = Matrix(cholesky(Σ).L)
logl!(obs, β, L, σ²)

-3261.3155007293885

**You will lose all 15 + 30 + 30 = 75 points** if the following statement throws `AssertionError`.

In [9]:
@assert logl!(obs, β, Matrix(cholesky(Σ).L), σ²) ≈ logpdf(mvn, y)

## Q4. Efficiency (30 pts)

Benchmarking your code and compare to the Distributions.jl function `logpdf`.

In [10]:
# benchmark the `logpdf` function in Distribution.jl
bm1 = @benchmark logpdf($mvn, $y)

BenchmarkTools.Trial: 6787 samples with 1 evaluation per sample.
 Range (min … max):  563.917 μs …  29.850 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     675.917 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   734.763 μs ± 451.104 μs  ┊ GC (mean ± σ):  0.19% ± 1.83%

   █▆▄▃▃▃▂▂▁                                                     
  ▄██████████▇▇▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  564 μs           Histogram: frequency by time         1.36 ms <

 Memory estimate: 32.12 KiB, allocs estimate: 6.

In [11]:
# benchmark your implementation
L = Matrix(cholesky(Σ).L)
bm2 = @benchmark logl!($obs, $β, $L, $σ²)

BenchmarkTools.Trial: 10000 samples with 200 evaluations per sample.
 Range (min … max):  399.375 ns …  2.194 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     411.460 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   424.491 ns ± 54.513 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃██▇▅▅▄▃▄▃▂▁                                                 ▂
  █████████████████▇▇▇▇▆▇▇▇▆▆▆▆▄▆▅▅▆▅▄▄▅▅▅▅▅▄▃▄▄▅▅▄▃▄▄▄▄▂▃▂▃▄▄ █
  399 ns        Histogram: log(frequency) by time       672 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

The points you will get is
$$
\frac{x}{1000} \times 30,
$$
where $x$ is the speedup of your program against the standard method.

In [12]:
# this is the points you'll get
clamp(median(bm1).time / median(bm2).time / 1000 * 30, 0, 30)

30.0

**Hint**: Apparently I am using 1000 as denominator because I expect your code to be at least $1000 \times$ faster than the standard method.

## Q5. Memory (30 pts)

You want to avoid memory allocation in the "hot" function `logl!`. You will lose 1 point for each `1 KiB = 1024 bytes` memory allocation. In other words, the points you get for this question is

In [13]:
clamp(30 - median(bm2).memory / 1024, 0, 30)

30.0

**Hint**: I am able to reduce the memory allocation to 0 bytes.

## Q6. Misc (15 pts)

Coding style, Git workflow, etc. For reproducibity, make sure we (TA and myself) can run your Jupyter Notebook. That is how we grade Q4 and Q5. If we cannot run it, you will get zero points.

## Q7. AI assistant (10 points)

Which AI assistants (e.g., GitHub Copilot) do you use when working on this assignment? Which AI model (e.g., GPT-5 mini, GPT-5, Claude Sonnet 4.5) does the AI assistant use? How do you use them? Do you think they help improve your productivity?

Give 5 instances where AI model gave incorrect or misleading answers. You can use screenshots or copy-paste the Q&A.

**Answer:**

I used Claude (claude.ai), specifically Claude Sonnet 4.6, throughout this assignment. I used it mainly for debugging Julia struct constructor errors (field order mismatches) and getting guidance on which BLAS routines to use and their correct flag arguments (trmm!, trsv!, potrf!). It helped me avoid spending hours reading BLAS documentation and quickly identified subtle bugs (like the contiguous memory issue with adjoints) that would have been hard to find by trial and error.

5 instances where the AI gave incorrect or misleading answers are as follows:

### Wrong scaling position in trmm

I asked for code to compute $\frac{1}{\sigma^2} L^\top (\mathbf{Z}^\top \mathbf{Z}) L$ and the AI initially wrote:
BLAS.trmm!('L', 'L', 'T', 'N', 1/σ², L, obs.storage_qq)  # wrong: scaling on left
BLAS.trmm!('R', 'L', 'N', 'N', 1.0,   L, obs.storage_qq)
This computed $\frac{1}{\sigma^2} L^\top (\mathbf{Z}^\top\mathbf{Z}) \cdot L$ rather than $L^\top(\mathbf{Z}^\top\mathbf{Z}) \cdot \frac{1}{\sigma^2}L$, which gives a wrong result since the scalar only affects the second factor. The correct code puts one(T) on the left multiply and one(T)/σ² on the right multiply.

### Suggested an unnecessary storage_n field

The AI originally suggested adding a storage_n :: Vector{T} of length $n_i$ to LmmObs to hold the residual $r = y - X\beta$ during logl!. This was unnecessary. Since we pre-compute $$y^\top y, \quad X^\top y, \quad Z^\top y, \quad X^\top X, \quad Z^\top X$$ in the constructor, all required inner products can be expanded algebraically: $$r^\top r = y^\top y - 2\beta^\top X^\top y + \beta^\top (X^\top X)\beta, \qquad Z^\top r = Z^\top y - (Z^\top X)\beta$$ so the $n_i$-vector $r$ never needs to be formed, keeping allocations at exactly 0 bytes.

### Used Symmetric() wrapper causing allocation

The AI suggested:
Mc = cholesky!(Symmetric(obs.storage_qq))
to compute the in-place Cholesky of $\mathbf{M} = \mathbf{I}_q + \frac{1}{\sigma^2}L^\top(\mathbf{Z}^\top\mathbf{Z})L$. The Symmetric() wrapper allocates a new object and cholesky! returns a Cholesky factorization struct — both cause heap allocations. The correct zero-allocation approach is:
LAPACK.potrf!('L', obs.storage_qq)
which factors $\mathbf{M}$ in-place with no allocations.

### diag(Mc.L) allocates a vector

Following from Instance 3, the AI computed $\log|\mathbf{V}i| = n_i\log\sigma^2 + \log|\mathbf{M}| = n_i\log\sigma^2 + 2\sum_j \log M{jj}$ using:
logdetV = 2 * sum(log, diag(Mc.L))
The diag() call allocates a fresh $q$-vector. The correct zero-allocation version reads diagonal entries directly:
@inbounds for j in 1:q
    logdetV += 2 * log(obs.storage_qq[j, j])
end

### Incorrect quadratic form expansion

When deriving $r^\top r$ without forming $r$, the AI initially produced logically inconsistent code:
copy!(obs.storage_p, obs.xty)             # storage_p = X'y
rtr = obs.yty
BLAS.symv!(...)                            # [forgot to overwrite storage_p with xtx*β]
rtr += dot(β, obs.storage_p) - 2*dot(β, obs.xty)
At the dot call, storage_p still held $X^\top y$ rather than $(X^\top X)\beta$, so it computed $\beta^\top(X^\top y) - 2\beta^\top(X^\top y) = -\beta^\top X^\top y$ instead of the correct $$r^\top r = y^\top y - 2\beta^\top X^\top y + \beta^\top(X^\top X)\beta.$$ The fix is to call BLAS.symv!('U', one(T), obs.xtx, β, zero(T), obs.storage_p) before taking the dot product, so that storage_p correctly holds $(X^\top X)\beta$.
